# TranscriptLab quickstart (CPU, about a minute, no GPU, no extra packages)

This notebook needs only the Python standard library. It shows four things:

1. **PNC**: loads a small sample of the public corpus of Gemma 4 agent trajectories.
2. **The parser on known-intent turns**: finds the ill-formed turns whose intended call is known, then runs
   slipguard's strict parser and its opt-in repair on them.
3. **`slipguard audit`**: scans an agent transcript for history hazards.
4. **The headline gains**: prints the numbers of the paper's Figure 1, read from the released result JSONs.

It looks for the files in this order:
- the repository: `$TRANSCRIPTLAB_ROOT`, or a parent of the working directory;
- the PNC dataset: `$PNC_ROOT`, then `<repo>/data/pnc`, then `/kaggle/input/*`.

In [ ]:
import glob, hashlib, json, os, subprocess, sys, time
from pathlib import Path

T0 = time.time()


def find_repo():
    for c in [os.environ.get("TRANSCRIPTLAB_ROOT"), *map(str, [Path.cwd(), *Path.cwd().parents]), *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/*/*")]:
        if c and Path(c, "experiments").is_dir() and (Path(c, "slipguard").is_dir() or Path(c, "release", "lib").is_dir()):
            return Path(c)
    raise SystemExit("repository not found: set TRANSCRIPTLAB_ROOT")


def find_pnc(repo):
    cands = [os.environ.get("PNC_ROOT"), repo / "data" / "pnc", *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/*/*")]
    for c in cands:
        if c and Path(c, "MANIFEST.json").exists() and Path(c, "episodes").is_dir():
            return Path(c)
    raise SystemExit("PNC dataset not found: set PNC_ROOT to the directory holding MANIFEST.json and episodes/")


REPO = find_repo()
PNC = find_pnc(REPO)
SG = REPO / "slipguard" if (REPO / "slipguard" / "src").is_dir() else REPO / "release" / "lib"   # public tree / project tree
try:
    import slipguard  # installed (pip install ./slipguard)
except ImportError:
    sys.path.insert(0, str(SG / "src"))
    import slipguard
OUT = Path(os.environ.get("QUICKSTART_OUT", "quickstart_out")).resolve()
OUT.mkdir(parents=True, exist_ok=True)
print("repository:", REPO, "\nPNC:", PNC, "\nslipguard", slipguard.__version__, "from", Path(slipguard.__file__).parent)

## 1. PNC at a glance

PNC holds 1,181 natural SWE-bench trajectories of `gemma-4-31b-it-qat-w4a16-ct`. Every turn records:
- the exact request;
- the raw generation, with the native `<|tool_call>` protocol;
- the calls exactly as vLLM 0.19.1 delivered them.

The SWE-bench grading of the final patches is in `grading.jsonl`, and `MANIFEST.json` pins its sha256.

In [ ]:
man = json.loads((PNC / "MANIFEST.json").read_text())
grading = PNC / "grading.jsonl"
sha = hashlib.sha256(grading.read_bytes()).hexdigest()
assert sha == man["files"]["grading.jsonl"], "grading.jsonl does not match the manifest"
rows = [json.loads(l) for l in grading.read_text().splitlines() if l.strip()]
episodes = sorted(Path(p).stem for p in glob.glob(str(PNC / "episodes" / "*.jsonl")))
print(f"{man['name']} ({man['version']})")
print(f"episodes: {man['n_episodes']:,} (files present: {len(episodes):,}); splits: {man['splits']['episodes_per_split_and_schema']}")
print(f"graded: {sum(r['graded'] for r in rows):,}; resolved: {sum(bool(r['resolved']) for r in rows):,} "
      f"(grading.jsonl sha256 {sha[:12]}... = MANIFEST.json#files/grading.jsonl)")
print(f"model: {man['model']['hf_repo']} @ {man['model']['revision'][:10]}; server: {man['server']['engine']} {man['server']['version']}")

### A small sample

The sample is every 30th episode in sorted order, 40 episodes in all. Each line of an episode file is one model turn.

In [ ]:
SAMPLE = episodes[::30]
turns = {}
for e in SAMPLE:
    with open(PNC / "episodes" / f"{e}.jsonl", encoding="utf-8") as f:
        turns[e] = sorted((json.loads(l) for l in f), key=lambda d: d["turn_index"])
n_turns = sum(len(v) for v in turns.values())
print(f"sample: {len(SAMPLE)} episodes, {n_turns:,} turns")
d = turns[SAMPLE[0]][1]
print("turn keys:", ", ".join(d))
raw = d["raw_completion"]
print("raw generation (call part):", raw[raw.find("<|tool_call>"):][:220] if "<|tool_call>" in raw else raw[:220])

## 2. The parser on turns with known intent

This step runs the corpus case builder that ships with slipguard (`conformance/build_pnc_cases.py`) on the sample.
For each ill-formed turn, the builder looks for a later retry within the next 3 turns that restates the call cleanly.
When it finds one, that retry is the turn's intended call.

Then we compare three ways of handling these turns:
- **built-in vLLM parsers**: not run here, because they need vLLM. Their result on all 218 such turns is in the
  released `RESULTS.json`;
- **slipguard strict**: delivers a call only if its text is strictly valid. Otherwise it surfaces an error, and the
  agent sees a loud failure instead of a wrong call;
- **slipguard opt-in repair**: a text-only repair that must reproduce the intended call exactly.

In [ ]:
sd = OUT / "pnc_sample"
(sd / "episodes").mkdir(parents=True, exist_ok=True)
for s in ("S1", "S2"):
    p = sd / "contexts" / f"P-swebench-31b-{s}-v1"
    p.mkdir(parents=True, exist_ok=True)
    if not (p / "manifest.json").exists():
        os.symlink(PNC / "contexts" / f"P-swebench-31b-{s}-v1" / "manifest.json", p / "manifest.json")
for e in SAMPLE:
    if not (sd / "episodes" / f"{e}.jsonl").exists():
        os.symlink(PNC / "episodes" / f"{e}.jsonl", sd / "episodes" / f"{e}.jsonl")
r = subprocess.run([sys.executable, str(SG / "conformance" / "build_pnc_cases.py"), str(sd), str(OUT / "sample_cases.jsonl")],
                   capture_output=True, text=True, env={**os.environ, "PYTHONPATH": str(Path(slipguard.__file__).parents[1])})
assert r.returncode == 0, r.stderr[-2000:]
print("builder:", r.stdout.strip())
cases = [json.loads(l) for l in (OUT / "sample_cases.jsonl").read_text().splitlines()]
known = [c for c in cases if c["subfamily"] == "ill_intent_known"]
print(f"{len(cases):,} turns -> {len(known)} ill-formed turns with known intent")

In [ ]:
from slipguard.parse import extract


def as_calls(calls):
    return json.dumps([{"name": c.name, "arguments": c.arguments} for c in calls], sort_keys=True)


tally = {"strict delivered a wrong call": 0, "strict surfaced the error": 0,
         "repair exact": 0, "repair wrong": 0, "repair surfaced": 0}
for c in known:
    want = json.dumps(c["intended"], sort_keys=True)
    st = extract(c["text"], c["tools"], parse_thought=True)                  # strict, per-call delivery
    if st.errors:
        tally["strict surfaced the error"] += 1
    elif as_calls(st.calls) != want:
        tally["strict delivered a wrong call"] += 1
    rp = extract(c["text"], c["tools"], parse_thought=True, repair=True)     # opt-in text-only repair
    if rp.errors and not rp.repairs:
        tally["repair surfaced"] += 1
    else:
        tally["repair exact" if as_calls(rp.calls) == want else "repair wrong"] += 1
print(f"sample, {len(known)} turns:", tally)

ex = known[0]
i = ex["text"].rfind("<|tool_call>")
print("\nexample:", ex["id"])
print("  model wrote: ", ex["text"][i:i + 160].replace("\n", "\\n"))
print("  intended:    ", json.dumps(ex["intended"])[:160])
print("  strict error:", extract(ex["text"], ex["tools"], parse_thought=True).errors[0])

CP = next(p for p in (SG / "results" / "conformance_pnc" / "RESULTS.json",) if p.exists())
res = json.loads(CP.read_text())
print(f"\nall {res['populations']['ill_intent_known']} known-intent PNC turns (RESULTS.json):")
for t, lab in (("vllm019.server", "vLLM 0.19.1 built-in"), ("vllm033.server", "vLLM 0.33.0 built-in"),
               ("gg.vllm033.server", "slipguard strict (vLLM plugin)"), ("gg.repair", "slipguard opt-in repair")):
    k = res["targets"][t]["intent_known"]
    print(f"  {lab:32s} wrong {k['wrong']:3d}  exact {k['exact']:3d}  surfaced {k['surfaced']:3d}  lost {k['lost']:3d}")

## 3. `slipguard audit` on a transcript

The transcript is the last request of a sample episode: the full history the model saw, in OpenAI message format,
written as the serving stack delivered it. `slipguard audit` reports the following, and changes nothing:
- historical calls that slipguard's quarantine would fix (`malformed_call`);
- raw call markup returned as text (`raw_call_in_content`);
- control-token text in tool data (`control_text`).

In [ ]:
def last_transcript(e):
    d = turns[e][-1]
    tools = json.loads((PNC / "contexts" / f"P-swebench-31b-{d['schema']}-v1" / "manifest.json").read_text())["tools"]
    return {"messages": d["request"]["messages"], "tools": tools}


from slipguard.audit import audit_messages

found = {}
for e in SAMPLE:
    t = last_transcript(e)
    found[e] = audit_messages(t["messages"], t["tools"], enable_thinking=True)
pick = next(e for e in SAMPLE if any(f["finding"] == "malformed_call" for f in found[e]))
path = OUT / f"transcript_{pick}.json"
path.write_text(json.dumps(last_transcript(pick)))
r = subprocess.run([sys.executable, "-m", "slipguard", "audit", str(path), "--thinking", "--json"], capture_output=True, text=True,
                   env={**os.environ, "PYTHONPATH": str(Path(slipguard.__file__).parents[1])})
print("$ slipguard audit", path.name, "--thinking --json")
print(r.stdout.strip()[:1500])
n_with = sum(1 for v in found.values() if any(f["finding"] == "malformed_call" for f in v))
print(f"\nsample: {n_with} of {len(SAMPLE)} final transcripts carry at least one call that quarantine would fix "
      f"({sum(f['finding'] == 'malformed_call' for v in found.values() for f in v)} calls in all)")

## 4. The headline gains (paper Figure 1), read from the released result JSONs

| row | stock | slipguard |
|---|---|---|
| end-to-end, ADK harness | stock serving | slipguard proxy |
| public turns | built-in vLLM parsers | slipguard |
| injected slips | lenient parser | repair (IPD) |
| history | laundered call + error | quarantine |
| checkpoint audit | QAT | AWQ pick, then quarantine |
| template | official | patched |

In [ ]:
def J(rel):
    return json.loads((REPO / rel).read_text())


def at(d, key):
    for k in key.split("/"):
        d = d[int(k)] if isinstance(d, list) else d[k]
    return d


E2E = J("experiments/endtoend/analysis/prereg_pop31.json")
V24 = J("experiments/intent_decoding/v24/battery_run_analysis.json")
V25 = J("experiments/intent_decoding/v25/out/analysis_v25.json")
V26 = J("experiments/contagion/out/confirmatory/v26_audit_analysis.json")
CF = J("experiments/contagion/out/confirmatory/confirm_v2_all_full.json")
pc = lambda x: f"{100 * x:.1f}%"
stuck = [at(E2E, f"secondary/stuck/{r}/{a}") for r in ("replicate1", "replicate2") for a in ("base", "fullp")]
wrong = [res["targets"][f"vllm0{v}.server"]["intent_known"]["wrong"] for v in ("19", "24", "31", "32", "33")]
rows = [
    ("malformed calls / episode (E2E)", f"{at(E2E, 'co_primary/Y1/base_mean'):.2f}", f"{at(E2E, 'co_primary/Y1/fullp_mean'):.2f}",
     "ratio {:.2f} [{:.2f}, {:.2f}]".format(at(E2E, "co_primary/Y1/ratio_fullp_over_base/ratio"), *at(E2E, "co_primary/Y1/ratio_fullp_over_base/ci95"))),
    ("stuck episodes (E2E)", f"{stuck[0][0] + stuck[2][0]}/{stuck[0][1] + stuck[2][1]}", f"{stuck[1][0] + stuck[3][0]}/{stuck[1][1] + stuck[3][1]}", "2 replicates pooled"),
    ("known-intent turns delivered wrong", f"{min(wrong)}-{max(wrong)}/{res['populations']['ill_intent_known']}",
     f"{res['targets']['gg.vllm033.server']['intent_known']['wrong']}/{res['populations']['ill_intent_known']}", "vLLM 0.19-0.33"),
    ("exact intent, injected slips", pc(at(V24, "outcomes_detectable/lenient/exact/reweighted/rate")),
     pc(at(V24, "outcomes_detectable/IPD/exact/reweighted/rate")),
     "IPD CI [{:.1f}, {:.1f}]".format(*[100 * x for x in at(V24, "outcomes_detectable/IPD/exact/reweighted/wilson95_kish")])),
    ("malformed next edit, laundered history", pc(at(V26, "rates/QAT/P1")[0]), pc(at(V26, "rates/QAT/P8a")[0]),
     "{:.0f}% of the excess removed [{:.0f}, {:.0f}]".format(100 * at(CF, "31B/full/C5/uncond/removal"), *[100 * x for x in at(CF, "31B/full/C5/uncond/CI")])),
    ("audit pick (QAT -> AWQ -> + quarantine)", pc(at(V26, "rates/QAT/P1")[0]), f"{pc(at(V26, 'rates/AWQ/P1')[0])} -> {pc(at(V26, 'rates/AWQ/P8a')[0])}",
     "RD {:+.1f} pp".format(100 * at(V26, "H1/RD"))),
    ("thinking-off ignored after a tool result", pc(at(V25, "F3/STOCK/rate")), pc(at(V25, "F3/GUARDED/rate")),
     f"{at(V25, 'F3/STOCK/k')}/{at(V25, 'F3/STOCK/n')} -> {at(V25, 'F3/GUARDED/k')}/{at(V25, 'F3/GUARDED/n')}"),
]
w = max(len(r[0]) for r in rows)
print(f"{'':{w}s}  {'stock':>12s}  {'slipguard':>18s}  note")
for r in rows:
    print(f"{r[0]:{w}s}  {r[1]:>12s}  {r[2]:>18s}  {r[3]}")
print(f"\nquickstart finished in {time.time() - T0:.1f} s")